# Fase 17E V7 — Hybrid Smoke Tests autocontidos

Execute **somente a célula operacional abaixo** em um runtime Colab recém-reiniciado.

Ela valida todos os pré-requisitos, preserva os checkpoints aprovados, executa apenas datasets pendentes e consolida automaticamente a Fase 17E. Resultados de smoke permanecem marcados como `usable_in_thesis=false`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import csv, gc, hashlib, json, os, re, shutil, subprocess, time
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

CAMPAIGN_ID='THESIS_OFFICIAL_CAMPAIGN_V2_20260901'; PROTOCOL_VERSION='phase17e_hybrid_smoke_v4_composite_numeric_gate'; SEED=42; NUM_CLIENTS=5
ROUNDS=1; LOCAL_EPOCHS=3; BATCH_SIZE=64; LEARNING_RATE=0.1
BASE=Path('/content/drive/MyDrive/Mestrado_Criptografia/OFFICIAL_CAMPAIGN_V2')
ROOT=BASE/CAMPAIGN_ID; CONTROL=ROOT/'00_CAMPAIGN_CONTROL'; FREEZE=ROOT/'01_DATASET_FREEZE'; SMOKE=ROOT/'03_SMOKE_TESTS'
EVIDENCE=ROOT/'00_CAMPAIGN_CONTROL'/'EXPORTS'; EVIDENCE.mkdir(parents=True,exist_ok=True)
EXPECTED_PARTITION_HASHES={'PHYSIONET_CHALLENGE_2012':'d175d06e19dcb0df7c668bf2184bfb93bf9e2d76cff1d8a430ec11ffa2fb7153','DAHL_RATS':'ebaba9d08cd6a5303bc739423893dbe6d62a951e2982c33b1b53173414a69a5d','CHEXCHONET':'e149adc9da9d9498b2df9c5cd6546f0d18ac5b732ff432a510c2e9f7f9281de7'}
EXPECTED_DATA_HASHES={'PHYSIONET_CHALLENGE_2012':'9b177585e87eee8bb201e9abd451a2a0bea80776773d1c0f45b50c54d2b1e76a','DAHL_RATS':'b0539e4a6ecd9eb36e0ae18379c9cd3787f1546a248eb00b78a9e19d10b90af7','CHEXCHONET':'12c7145628e25bd389cb642c38d19f8c4e8927c8d2f5b2182d7fe215dabd60a4'}
def utc(): return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')
def sha256_file(p,chunk=8<<20):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  while True:
   b=f.read(chunk)
   if not b: break
   h.update(b)
 return h.hexdigest()
print('CAMPAIGN=',ROOT); print('EXECUTION=HYBRID_SMOKE_R1_C5; usable_in_thesis=false')

import ctypes, psutil
gc.collect()
try:
 ctypes.CDLL('libc.so.6').malloc_trim(0)
except Exception:
 pass
memory=psutil.virtual_memory(); disk=shutil.disk_usage('/content')
print('='*110)
print('FASE17E_V7_AUTOCONTIDA — VERIFICAÇÃO DE RECURSOS')
print(f'RAM disponível: {memory.available/1024**3:.2f} GiB')
print(f'Disco livre: {disk.free/1024**3:.2f} GiB')
assert memory.available >= 8*1024**3, 'RAM disponível inferior a 8 GiB; reinicie o runtime.'
assert disk.free >= 8*1024**3, 'Espaço livre inferior a 8 GiB.'
os.environ['GOGC']='20'
# Go 1.18 does not implement GOMEMLIMIT. Remove stale values inherited from
# earlier diagnostic cells so execution is reproducible.
os.environ.pop('GOMEMLIMIT',None)
print('RESOURCE_GATE=PASS')


required={'17B1':CONTROL/'PHASE17B1_MASTER_GATE.json','17C':CONTROL/'PHASE17C_MASTER_GATE.json','17D':CONTROL/'PHASE17D_MASTER_GATE.json','17E0':CONTROL/'PHASE17E0_MASTER_GATE.json'}
gates={}
for k,p in required.items():
 assert p.exists(),f'Gate ausente: {p}'; gates[k]=json.loads(p.read_text(encoding='utf-8'))
assert gates['17B1']['five_clients_noniid_frozen'] is True
assert gates['17C']['all_baseline_smokes_approved'] is True
assert gates['17D']['all_ckks_smokes_approved'] is True
assert gates['17E0']['approved'] is True
assert gates['17E0']['decryptions_before_aggregation']==0
assert gates['17E0']['next_authorized_step']=='FASE_17E_HYBRID_SMOKE_TESTS'
print('ALL_PREREQUISITE_GATES=PASS')


REPO=Path('/content/RtF-Transciphering'); COMMIT='105fc73115b56f1d6ff357029c7682b19a6d8510'
subprocess.run(['apt-get','update','-qq'],check=True); subprocess.run(['apt-get','install','-y','-qq','golang-go','git'],check=True)
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/KAIST-CryptLab/RtF-Transciphering.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','--all','--tags'],check=True); subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==COMMIT
GO_SOURCE='package ckks_fv\n\nimport (\n    "crypto/rand"\n    "encoding/json"\n    "fmt"\n    "math"\n    "os"\n    "runtime"\n    "runtime/debug"\n    "testing"\n    "time"\n    "github.com/ldsec/lattigo/v2/utils"\n)\n\ntype phase17E0Request struct {\n    Dataset string `json:"dataset"`\n    OriginalLength int `json:"original_length"`\n    ClientDeltas [][]float64 `json:"client_deltas"`\n    ClientWeights []float64 `json:"client_weights"`\n}\ntype phase17E0Response struct {\n    Dataset string `json:"dataset"`\n    OriginalLength int `json:"original_length"`\n    Clients int `json:"clients"`\n    RecoveredAggregate []float64 `json:"recovered_aggregate"`\n    MaxAbsError float64 `json:"max_abs_error"`\n    MeanAbsError float64 `json:"mean_abs_error"`\n    MaxPaddingError float64 `json:"max_padding_error"`\n    WallSeconds float64 `json:"wall_seconds"`\n    RubatoVariant string `json:"rubato_variant"`\n    BridgeVersion string `json:"bridge_version"`\n    AggregatedBeforeDecrypt bool `json:"aggregated_before_decrypt"`\n    ClientDecryptions int `json:"client_decryptions"`\n}\n\nfunc TestPhase17E0EncryptedFedAvg(t *testing.T) {\n    raw := os.Getenv("PHASE17E0_REQUEST_JSON")\n    if raw == "" { t.Fatal("PHASE17E0_REQUEST_JSON ausente") }\n    var req phase17E0Request\n    if err := json.Unmarshal([]byte(raw), &req); err != nil { t.Fatal(err) }\n    if len(req.ClientDeltas) != 5 || len(req.ClientWeights) != 5 { t.Fatal("expected exactly 5 clients and 5 weights") }\n    if req.OriginalLength < 1 || req.OriginalLength > 513 { t.Fatal("original_length must be in [1,513]") }\n    sumW := 0.0\n    for i := range req.ClientDeltas {\n        if len(req.ClientDeltas[i]) != req.OriginalLength { t.Fatalf("client %d: expected %d values, got %d", i, req.OriginalLength, len(req.ClientDeltas[i])) }\n        if req.ClientWeights[i] <= 0 { t.Fatalf("client %d: weight must be positive", i) }\n        sumW += req.ClientWeights[i]\n    }\n    if math.Abs(sumW-1.0) > 1e-12 { t.Fatalf("weights must sum to 1; got %.17g", sumW) }\n\n    rubatoParam := RUBATO80S\n    blocksize := RubatoParams[rubatoParam].Blocksize\n    numRound := RubatoParams[rubatoParam].NumRound\n    plainModulus := RubatoParams[rubatoParam].PlainModulus\n    sigma := RubatoParams[rubatoParam].Sigma\n    hbtpParams := RtFRubatoParams[0]\n    params, err := hbtpParams.Params(); if err != nil { t.Fatal(err) }\n    params.SetPlainModulus(plainModulus); params.SetLogFVSlots(params.LogN())\n    messageScaling := float64(params.PlainModulus()) / hbtpParams.MessageRatio\n    rubatoModDown := RubatoModDownParams[rubatoParam].CipherModDown\n    stcModDown := RubatoModDownParams[rubatoParam].StCModDown\n    kgen := NewKeyGenerator(params); sk, pk := kgen.GenKeyPairSparse(hbtpParams.H)\n    fvEncoder := NewMFVEncoder(params); ckksEncoder := NewCKKSEncoder(params)\n    fvEncryptor := NewMFVEncryptorFromPk(params, pk); ckksDecryptor := NewCKKSDecryptor(params, sk)\n    rotationsHalfBoot := kgen.GenRotationIndexesForHalfBoot(params.LogSlots(), hbtpParams)\n    pDcds := fvEncoder.GenSlotToCoeffMatFV(2)\n    rotations := append(rotationsHalfBoot, kgen.GenRotationIndexesForSlotsToCoeffsMat(pDcds)...)\n    rotkeys := kgen.GenRotationKeysForRotations(rotations, true, sk); rlk := kgen.GenRelinearizationKey(sk)\n    evk := EvaluationKey{Rlk: rlk, Rtks: rotkeys}\n    hbtp, err := NewHalfBootstrapper(params, hbtpParams, BootstrappingKey{Rlk: rlk, Rtks: rotkeys}); if err != nil { t.Fatal(err) }\n    fvEvaluator := NewMFVEvaluator(params, evk, pDcds)\n    ckksEvaluator := NewCKKSEvaluator(params, evk)\n    key := make([]uint64, blocksize); for i := range key { key[i] = uint64(i+1) }\n    keyRubato := NewMFVRubato(rubatoParam, params, fvEncoder, fvEncryptor, fvEvaluator, rubatoModDown[0])\n    kCt := keyRubato.EncKey(key)\n\n    start := time.Now()\n    expected := make([]float64, 513)\n    var encryptedAggregate *Ciphertext\n    for clientID := 0; clientID < 5; clientID++ {\n        fmt.Printf("PHASE17E0_PROGRESS dataset=%s client=%d/5 stage=transciphering\\n", req.Dataset, clientID+1)\n        data := make([]float64, params.N()); copy(data[:req.OriginalLength], req.ClientDeltas[clientID])\n        for j := 0; j < req.OriginalLength; j++ { expected[j] += req.ClientWeights[clientID] * req.ClientDeltas[clientID][j] }\n        nonces := make([][]byte, params.N()); keystream := make([][]uint64, params.N())\n        for i := 0; i < params.N(); i++ { nonces[i] = make([]byte,8); if _,err=rand.Read(nonces[i]);err!=nil{t.Fatal(err)} }\n        counter := make([]byte,8); if _,err=rand.Read(counter);err!=nil{t.Fatal(err)}\n        for i := 0; i < params.N(); i++ { keystream[i] = plainRubato(blocksize,numRound,nonces[i],counter,key,plainModulus,sigma) }\n        coeffs := make([]float64, params.N())\n        for i := 0; i < params.N()/2; i++ { j:=utils.BitReverse64(uint64(i),uint64(params.LogN()-1)); coeffs[j]=data[i]; coeffs[j+uint64(params.N()/2)]=data[i+params.N()/2] }\n        plainRingT := ckksEncoder.EncodeCoeffsRingTNew(coeffs,messageScaling); poly:=plainRingT.Value()[0]\n        for i := 0; i < params.N(); i++ { j:=utils.BitReverse64(uint64(i),uint64(params.LogN())); poly.Coeffs[0][j]=(poly.Coeffs[0][j]+keystream[i][0])%params.PlainModulus() }\n        plaintext:=NewPlaintextFVLvl(params,0); fvEncoder.FVScaleUp(plainRingT,plaintext)\n        // A fresh Rubato workspace per client prevents mutable modulus-chain state from leaking across clients.\n        clientRubato:=NewMFVRubato(rubatoParam,params,fvEncoder,fvEncryptor,fvEvaluator,rubatoModDown[0])\n        fvKeystreams:=clientRubato.Crypt(nonces,counter,kCt,rubatoModDown); fvKS:=fvEvaluator.SlotsToCoeffs(fvKeystreams[0],stcModDown)\n        level:=fvKS.Level()\n        if level>0 { fvEvaluator.ModSwitchMany(fvKS,fvKS,level) }\n        ciphertext:=NewCiphertextFVLvl(params,1,0); ciphertext.Value()[0]=plaintext.Value()[0].CopyNew(); fvEvaluator.Sub(ciphertext,fvKS,ciphertext); fvEvaluator.TransformToNTT(ciphertext,ciphertext)\n        ciphertext.SetScale(math.Exp2(math.Round(math.Log2(float64(params.Qi()[0])/float64(params.PlainModulus())*messageScaling))))\n        ctBoot,_:=hbtp.HalfBoot(ciphertext,false)\n        weighted:=ckksEvaluator.MultByConstNew(ctBoot,req.ClientWeights[clientID])\n        if encryptedAggregate==nil { encryptedAggregate=weighted } else { ckksEvaluator.Add(encryptedAggregate,weighted,encryptedAggregate) }\n        fmt.Printf("PHASE17E0_PROGRESS dataset=%s client=%d/5 stage=encrypted_aggregate_updated\\n", req.Dataset, clientID+1)\n        // Release all client-local material before the next transciphering. Only the CKKS aggregate survives.\n        data=nil; nonces=nil; keystream=nil; coeffs=nil; plainRingT=nil; plaintext=nil\n        clientRubato=nil; fvKeystreams=nil; fvKS=nil; ciphertext=nil; ctBoot=nil; weighted=nil\n        runtime.GC(); debug.FreeOSMemory()\n        fmt.Printf("PHASE17E0_PROGRESS dataset=%s client=%d/5 stage=memory_released\\n", req.Dataset, clientID+1)\n    }\n    // SECURITY BOUNDARY: the sole decrypt operation is after all five CKKS ciphertexts were aggregated.\n    values:=ckksEncoder.DecodeComplex(ckksDecryptor.DecryptNew(encryptedAggregate),params.LogSlots())\n    recovered:=make([]float64,req.OriginalLength); maxErr:=0.0; meanErr:=0.0; maxPad:=0.0\n    for i:=0;i<req.OriginalLength;i++ { recovered[i]=real(values[i]); e:=math.Abs(recovered[i]-expected[i]); meanErr+=e; if e>maxErr{maxErr=e} }\n    meanErr/=float64(req.OriginalLength)\n    for i:=req.OriginalLength;i<513;i++ { e:=math.Abs(real(values[i])); if e>maxPad{maxPad=e} }\n    resp:=phase17E0Response{req.Dataset,req.OriginalLength,5,recovered,maxErr,meanErr,maxPad,time.Since(start).Seconds(),"RUBATO80S","phase17e0_encrypted_fedavg_v1",true,0}\n    b,err:=json.Marshal(resp);if err!=nil{t.Fatal(err)};fmt.Printf("PHASE17E0_JSON:%s\\n",b)\n}\n'
GO_FILE=REPO/'ckks_fv'/'phase17e0_encrypted_fedavg_test.go'; GO_FILE.write_text(GO_SOURCE,encoding='utf-8')
assert GO_SOURCE.count('DecryptNew(')==1 and GO_SOURCE.index('DecryptNew(')>GO_SOURCE.index('SECURITY BOUNDARY')
assert 'clientRubato:=NewMFVRubato' in GO_SOURCE and 'RecoveredDelta' not in GO_SOURCE
bridge_sha=hashlib.sha256(GO_SOURCE.encode()).hexdigest(); subprocess.run(['gofmt','-w',str(GO_FILE)],check=True)
p=subprocess.run(['go','test','./ckks_fv','-run','^$','-count=1'],cwd=REPO,text=True,capture_output=True)
print(p.stdout,p.stderr); assert p.returncode==0,'Bridge não compilou.'
print('BRIDGE_COMPILE_AND_STATIC_SECURITY_GATE=PASS'); print('BRIDGE_SHA256=',bridge_sha)


def load_npz_indices(path):
 z=np.load(path,allow_pickle=False); keys=list(z.files)
 def pick(cands):
  for k in cands:
   if k in z: return np.asarray(z[k],dtype=np.int64)
  raise KeyError(f'Nenhuma chave {cands}; disponíveis={keys}')
 return pick(['train','train_idx','idx_train']),pick(['validation','val','validation_idx','val_idx','idx_val']),pick(['test','test_idx','idx_test'])

def load_clients(path):
 assert sha256_file(path)==EXPECTED_PARTITION_HASHES[path.parts[-3]],f'Hash da partição divergente: {path}'
 z=np.load(path,allow_pickle=False); out=[]
 for i in range(5):
  found=None
  for k in [f'client_{i}',f'client{i}',f'client_{i}_indices',f'client{i}_indices',str(i)]:
   if k in z: found=np.asarray(z[k],dtype=np.int64); break
  if found is None: raise KeyError(f'Cliente {i} ausente; chaves={z.files}')
  out.append(found)
 return out

DAHL_FEATURES=['mean','std','min','max','median','q05','q25','q75','q95','rms','range','mean_abs_diff']
def load_dataset(name):
 d=FREEZE/name/'SCIENTIFIC_FREEZE'; part=d/'official_client_partition_noniid_seed42.npz'
 clients=load_clients(part)
 if name=='PHYSIONET_CHALLENGE_2012':
  data=d/'physionet_challenge_2012_features.csv'; assert sha256_file(data)==EXPECTED_DATA_HASHES[name]
  df=pd.read_csv(data); features=[c for c in df.columns if c not in ['RecordID','target']]
  assert len(features)==265; X=df[features].to_numpy(np.float64); y=df['target'].to_numpy(np.int64)
  tr,va,te=load_npz_indices(d/'official_split_seed42.npz'); init=np.asarray(np.load(d/'official_initial_state.npy'),dtype=np.float64).reshape(-1)
 elif name=='DAHL_RATS':
  data=d/'dahl_derived_features.csv'; assert sha256_file(data)==EXPECTED_DATA_HASHES[name]
  df=pd.read_csv(data); assert all(c in df for c in DAHL_FEATURES)
  X=df[DAHL_FEATURES].to_numpy(np.float64); y=df['target'].to_numpy(np.int64); tr,va,te=load_npz_indices(d/'official_split_seed42.npz'); init=np.zeros(13,np.float64)
 else:
  data=d/'chexchonet_embeddings.npz'; assert sha256_file(data)==EXPECTED_DATA_HASHES[name]
  z=np.load(data,allow_pickle=False); X=np.asarray(z['embeddings'],dtype=np.float64); y=np.asarray(z['targets'],dtype=np.int64)
  raw=np.load(d/'official_split_original.npy',allow_pickle=True)
  def split_label(x):
   if isinstance(x,(bytes,np.bytes_)): x=x.decode('utf-8')
   s=str(x).strip().lower()
   return 'validation' if s in {'val','valid','validation'} else s
  norm=np.array([split_label(x) for x in raw]); tr=np.where(norm=='train')[0]; va=np.where(norm=='validation')[0]; te=np.where(norm=='test')[0]; init=np.zeros(513,np.float64)
 assert len(clients)==5 and set(np.concatenate(clients).tolist())==set(tr.tolist())
 assert not (set(tr)&set(va) or set(tr)&set(te) or set(va)&set(te))
 X[~np.isfinite(X)]=np.nan
 fill=np.nanmedian(X[tr],axis=0); all_missing=~np.isfinite(fill); fill[all_missing]=0.0
 miss_r,miss_c=np.where(np.isnan(X)); X[miss_r,miss_c]=fill[miss_c]
 mu=X[tr].mean(0); sd=X[tr].std(0); sd[~np.isfinite(sd)|(sd<1e-12)]=1.0; X=(X-mu)/sd
 assert init.ndim==1 and init.size==X.shape[1]+1,f'{name}: estado inicial {init.shape}/{init.size}, esperado {X.shape[1]+1}'
 assert np.isfinite(init).all(),f'{name}: estado inicial contém valores não finitos'
 assert np.isfinite(X).all(),f'{name}: matriz de features contém {int((~np.isfinite(X)).sum())} valores não finitos'
 preprocessing_vector=np.r_[fill,mu,sd]
 return X,y,tr,va,te,clients,init,sha256_file(part),hashlib.sha256(np.ascontiguousarray(preprocessing_vector).tobytes()).hexdigest()

def sigmoid(x):
 x=np.clip(x,-40,40); return 1/(1+np.exp(-x))
def local_sgd(state,X,y,idx,client_id):
 w=state[:-1].copy(); b=float(state[-1]); t0=time.perf_counter()
 for epoch in range(LOCAL_EPOCHS):
  order=np.asarray(idx).copy(); np.random.default_rng(SEED+1000*client_id+epoch).shuffle(order)
  for s in range(0,len(order),BATCH_SIZE):
   q=order[s:s+BATCH_SIZE]; p=sigmoid(X[q]@w+b); err=p-y[q]
   w-=LEARNING_RATE*(X[q].T@err/len(q)); b-=LEARNING_RATE*float(err.mean())
 out=np.r_[w,b]; return out-state,time.perf_counter()-t0
def metrics(state,X,y,idx):
 p=sigmoid(X[idx]@state[:-1]+state[-1]); pred=(p>=.5).astype(int); tn,fp,fn,tp=confusion_matrix(y[idx],pred,labels=[0,1]).ravel()
 return {'accuracy':float(accuracy_score(y[idx],pred)),'precision':float(precision_score(y[idx],pred,zero_division=0)),'recall':float(recall_score(y[idx],pred,zero_division=0)),'f1':float(f1_score(y[idx],pred,zero_division=0)),'auroc':float(roc_auc_score(y[idx],p)),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)},p


DATASETS=['PHYSIONET_CHALLENGE_2012','DAHL_RATS','CHEXCHONET']; results=[]
for name in DATASETS:
 print('='*110); print('INICIANDO HYBRID SMOKE:',name,'— treino real, 1 rodada, 5 clientes')
 scenario_root=SMOKE/name/'HYBRID'; scenario_root.mkdir(parents=True,exist_ok=True)
 checkpoint=scenario_root/'LATEST_APPROVED_SMOKE.json'
 if checkpoint.exists():
  saved=json.loads(checkpoint.read_text(encoding='utf-8'))
  if saved.get('approved') is True and saved.get('protocol_version')==PROTOCOL_VERSION and saved.get('bridge_source_sha256')==bridge_sha and saved.get('partition_sha256')==EXPECTED_PARTITION_HASHES[name] and bool(saved.get('checks')) and all(bool(v) for v in saved['checks'].values()):
   print('CHECKPOINT APROVADO REUTILIZADO:',saved['run_id']); results.append(saved); continue
 X,y,tr,va,te,clients,state,part_hash,prep_hash=load_dataset(name); n=len(state)
 run_id=f"RUN__{name}__HYBRID__SMOKE__R1__C5__S42__{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"; run_dir=scenario_root/run_id; run_dir.mkdir(parents=True)
 started=utc(); rss0=__import__('psutil').Process().memory_info().rss; round_t0=time.perf_counter(); child_cpu0=__import__('resource').getrusage(__import__('resource').RUSAGE_CHILDREN)
 deltas=[]; client_rows=[]
 for cid,idx in enumerate(clients):
  delta,secs=local_sgd(state,X,y,idx,cid); assert np.isfinite(delta).all(); deltas.append(delta)
  client_rows.append({'client_id':cid,'samples':len(idx),'train_seconds':secs,'delta_l2_norm':float(np.linalg.norm(delta))})
  print(f'LOCAL_TRAIN dataset={name} client={cid+1}/5 samples={len(idx)} seconds={secs:.3f}',flush=True)
 weights=np.array([len(x) for x in clients],np.float64); weights/=weights.sum(); clear_agg=np.average(np.stack(deltas),axis=0,weights=weights)
 validation_X=np.ascontiguousarray(X[va],dtype=np.float64); validation_y=np.ascontiguousarray(y[va],dtype=np.int64); validation_idx=np.arange(len(va),dtype=np.int64)
 clear_state=state+clear_agg; clear_metrics,clear_prob=metrics(clear_state,validation_X,validation_y,validation_idx)
 req={'dataset':name,'original_length':n,'client_deltas':[x.tolist() for x in deltas],'client_weights':weights.tolist()}
 del X,y; gc.collect()
 try: __import__('ctypes').CDLL('libc.so.6').malloc_trim(0)
 except Exception: pass
 rss_before_bridge=__import__('psutil').Process().memory_info().rss
 print(f'MEMORY_RELEASED_BEFORE_BRIDGE dataset={name} rss_bytes={rss_before_bridge}',flush=True)
 env=os.environ.copy(); env.pop('GOMEMLIMIT',None); env['PHASE17E0_REQUEST_JSON']=json.dumps(req,separators=(',',':')); env['GOGC']='20'
 log_path=run_dir/'hybrid_bridge.log'; local_log_path=Path('/content')/f'{run_id}__hybrid_bridge.log'; tail=deque(maxlen=120); payload=None
 with local_log_path.open('w',encoding='utf-8') as lf:
  proc=subprocess.Popen(['go','test','./ckks_fv','-run','^TestPhase17E0EncryptedFedAvg$','-count=1','-v','-timeout=0'],cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
  for line in proc.stdout:
   line=line.rstrip(); lf.write(line+'\n'); lf.flush(); tail.append(line)
   if 'PHASE17E0_PROGRESS' in line or '--- FAIL' in line: print(line,flush=True)
   if 'PHASE17E0_JSON:' in line: payload=json.loads(line.split('PHASE17E0_JSON:',1)[1])
  rc=proc.wait()
 if rc!=0 or payload is None:
  print('\n'.join(tail))
  try:
   shutil.copy2(local_log_path,log_path)
  except OSError as drive_error:
   print('DRIVE_RECONNECT_REQUIRED_DURING_FAILURE_SAVE:',repr(drive_error))
   drive.mount('/content/drive',force_remount=True); run_dir.mkdir(parents=True,exist_ok=True); shutil.copy2(local_log_path,log_path)
  (run_dir/'RUN_STATUS.json').write_text(json.dumps({'run_id':run_id,'campaign_id':CAMPAIGN_ID,'status':'FAILED_BRIDGE','smoke_approved':False,'usable_in_thesis':False,'return_code':rc,'local_log':str(local_log_path),'drive_log':str(log_path)},indent=2),encoding='utf-8')
  raise RuntimeError(f'Bridge híbrido falhou: {name}, rc={rc}; log preservado em {log_path}')
 try:
  shutil.copy2(local_log_path,log_path)
 except OSError as drive_error:
  print('DRIVE_RECONNECT_REQUIRED:',repr(drive_error)); drive.mount('/content/drive',force_remount=True); run_dir.mkdir(parents=True,exist_ok=True); shutil.copy2(local_log_path,log_path)
 recovered=np.asarray(payload['recovered_aggregate'],np.float64); hybrid_state=state+recovered; hybrid_metrics,hybrid_prob=metrics(hybrid_state,validation_X,validation_y,validation_idx)
 err=np.abs(recovered-clear_agg); pred_dis=float(np.mean((clear_prob>=.5)!=(hybrid_prob>=.5)))
 nonbias_max=float(err[:-1].max()) if n>1 else 0.0; bias_abs=float(err[-1]); bias_relative=bias_abs/max(abs(float(clear_agg[-1])),1e-12); relative_l2=float(np.linalg.norm(recovered-clear_agg)/max(np.linalg.norm(clear_agg),1e-12)); max_padding=float(payload['max_padding_error'])
 tolerance_policy={'nonbias_max_abs_error':1e-3,'bias_relative_error':2e-3,'global_relative_l2_error':2e-3,'mean_abs_error':1e-4,'validation_probability_max_abs_difference':1e-3,'prediction_disagreement_rate':1e-3,'max_padding_error':1e-3}
 checks={'five_clients_completed':len(deltas)==5,'parameter_count_preserved':len(recovered)==n,'aggregated_before_decrypt':payload['aggregated_before_decrypt'] is True,'zero_client_decryptions':payload['client_decryptions']==0,'finite_decryption':bool(np.isfinite(recovered).all()),'nonbias_max_abs_error_le_1e_3':nonbias_max<=1e-3,'bias_relative_error_le_2e_3':bias_relative<=2e-3,'global_relative_l2_error_le_2e_3':relative_l2<=2e-3,'mean_abs_error_le_1e_4':float(err.mean())<=1e-4,'validation_probability_difference_le_1e_3':float(np.max(np.abs(clear_prob-hybrid_prob)))<=1e-3,'prediction_disagreement_le_1e_3':pred_dis<=1e-3,'max_padding_error_le_1e_3':max_padding<=1e-3,'test_not_used':True,'partition_hash_preserved':part_hash==EXPECTED_PARTITION_HASHES[name]}
 approved=all(checks.values()); completed=utc(); rss1=__import__('psutil').Process().memory_info().rss
 child_cpu1=__import__('resource').getrusage(__import__('resource').RUSAGE_CHILDREN)
 communication={'model_download_bytes':int(5*n*8),'five_client_update_plain_equivalent_bytes':int(5*n*8),'measurement_scope':'logical_payload_equivalent; bridge executed in-process without network serialization','measured_network_bytes':None}
 result={'dataset':name,'scenario':'HYBRID','execution_type':'SMOKE','run_id':run_id,'run_dir':str(run_dir),'approved':approved,'usable_in_thesis':False,'rounds':1,'clients':5,'local_epochs':LOCAL_EPOCHS,'batch_size':BATCH_SIZE,'learning_rate':LEARNING_RATE,'parameter_count':n,'partition_sha256':part_hash,'preprocessing_sha256':prep_hash,'imputation_policy':'training_median_per_feature; fallback_zero_only_for_all-missing-training-columns; fitted_on_train_only','bridge_source_sha256':bridge_sha,'bridge_commit':COMMIT,'rubato_variant':'RUBATO80S','aggregation':'Rubato_to_CKKS_encrypted_weighted_FedAvg','clear_reference_validation':clear_metrics,'hybrid_validation':hybrid_metrics,'tolerance_policy':tolerance_policy,'numeric_error':{'max_abs_error':float(err.max()),'nonbias_max_abs_error':nonbias_max,'bias_abs_error':bias_abs,'bias_relative_error':bias_relative,'global_relative_l2_error':relative_l2,'mean_abs_error':float(err.mean()),'median_abs_error':float(np.median(err)),'p95_abs_error':float(np.quantile(err,.95)),'p99_abs_error':float(np.quantile(err,.99)),'max_padding_error':max_padding,'rmse':float(np.sqrt(np.mean(err**2))),'validation_probability_max_abs_difference':float(np.max(np.abs(clear_prob-hybrid_prob))),'validation_prediction_disagreement_rate':pred_dis},'communication':communication,'hybrid_bridge_wall_seconds':payload['wall_seconds'],'round_wall_seconds':time.perf_counter()-round_t0,'child_cpu_seconds':float((child_cpu1.ru_utime+child_cpu1.ru_stime)-(child_cpu0.ru_utime+child_cpu0.ru_stime)),'rss_start_bytes':rss0,'rss_before_bridge_bytes':rss_before_bridge,'rss_end_bytes':rss1,'checks':checks,'clients_detail':client_rows,'started_at_utc':started,'completed_at_utc':completed}
 result['protocol_version']=PROTOCOL_VERSION
 (run_dir/'RUN_CONFIG.json').write_text(json.dumps({k:result[k] for k in ['dataset','scenario','execution_type','protocol_version','rounds','clients','local_epochs','batch_size','learning_rate','parameter_count','partition_sha256','preprocessing_sha256','imputation_policy','bridge_source_sha256','bridge_commit','rubato_variant','aggregation','tolerance_policy','usable_in_thesis']},indent=2),encoding='utf-8')
 (run_dir/'round_001_metrics.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
 (run_dir/'RUN_STATUS.json').write_text(json.dumps({'run_id':run_id,'campaign_id':CAMPAIGN_ID,'status':'COMPLETED_APPROVED' if approved else 'COMPLETED_REJECTED','smoke_approved':approved,'usable_in_thesis':False,'checks':checks},indent=2),encoding='utf-8')
 pd.DataFrame(client_rows).to_csv(run_dir/'client_metrics.csv',index=False)
 np.save(run_dir/'global_state_after_round_001.npy',hybrid_state,allow_pickle=False)
 result['result_sha256']=sha256_file(run_dir/'round_001_metrics.json')
 if approved: checkpoint.write_text(json.dumps(result,indent=2),encoding='utf-8')
 print(json.dumps({'dataset':name,'approved':approved,'validation':hybrid_metrics,'numeric_error':result['numeric_error'],'wall_seconds':result['round_wall_seconds'],'run_dir':str(run_dir)},indent=2)); assert approved,f'Smoke híbrido reprovado: {name}'
 results.append(result); del validation_X,validation_y,deltas; gc.collect()


all_ok=len(results)==3 and all(r['approved'] for r in results)
master={'phase':'17E','campaign_id':CAMPAIGN_ID,'completed_at_utc':utc(),'hybrid_smoke_approvals':{r['dataset']:r['approved'] for r in results},'all_hybrid_smokes_approved':all_ok,'smoke_results_usable_in_thesis':False,'official_campaign_authorized':all_ok,'next_authorized_step':'FASE_18_OFFICIAL_CAMPAIGN_30_ROUNDS_3_DATASETS_3_SCENARIOS' if all_ok else None,'results':results}
master_path=CONTROL/'PHASE17E_MASTER_GATE.json'; master_path.write_text(json.dumps(master,indent=2),encoding='utf-8')
status_path=CONTROL/'CAMPAIGN_STATUS.json'; status=json.loads(status_path.read_text(encoding='utf-8')); status.update({'status':'PHASE17E_COMPLETED','usable_in_thesis':False,'phase17e_hybrid_smokes_approved':all_ok,'official_campaign_authorized':all_ok,'next_authorized_step':master['next_authorized_step']}); status_path.write_text(json.dumps(status,indent=2),encoding='utf-8')
export=Path('/content/PHASE17E_EVIDENCE'); shutil.rmtree(export,ignore_errors=True); export.mkdir()
shutil.copy2(master_path,export/'PHASE17E_MASTER_GATE.json'); shutil.copy2(status_path,export/'CAMPAIGN_STATUS.json'); shutil.copy2(CONTROL/'PHASE17E0_MASTER_GATE.json',export/'PHASE17E0_MASTER_GATE.json')
for r in results:
 src=Path(r['run_dir']); dst=export/r['dataset']; dst.mkdir()
 for f in ['RUN_CONFIG.json','RUN_STATUS.json','round_001_metrics.json','client_metrics.csv','hybrid_bridge.log']:
  shutil.copy2(src/f,dst/f)
zip_path=Path(shutil.make_archive('/content/PHASE17E_EVIDENCE','zip','/content','PHASE17E_EVIDENCE')); drive_zip=EVIDENCE/'PHASE17E_EVIDENCE.zip'; shutil.copy2(zip_path,drive_zip)
print('='*110); print(json.dumps({'PHASE17E_APPROVED':all_ok,'official_campaign_authorized':all_ok,'smoke_results_usable_in_thesis':False,'next_authorized_step':master['next_authorized_step'],'gate':str(master_path),'evidence_zip':str(drive_zip)},indent=2)); print('='*110)


## Resultado esperado

Ao final: `PHASE17E_APPROVED=true`, os três checkpoints híbridos validados e o pacote `PHASE17E_EVIDENCE.zip` salvo na pasta de resultados da campanha.
